# Step 10 — Ten required observability experiments

This notebook runs all ten required experiments against both NammaMart pipeline implementations. The functions imported below are the same business-logic modules used by the ETL and ELT DAG wrappers.

Every experiment uses a temporary copy of the source files. The original CSV files are never edited, so restoration is automatic after each experiment.

In [5]:
import sys

print(sys.executable)

/Users/kumarsamanvaya/Documents/IITR - FDE Course/Data Pipepline Assignment/_Samy_IIT_Roorkie_CEC_Futurenece_Data_Pipeline_to_ML_Model/.venv/bin/python


In [6]:
# Install the notebook dependencies, including tabulate used by the report writer's
# DataFrame.to_markdown() call. Using %pip installs into the active notebook kernel.
%pip install pandas duckdb tabulate

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import duckdb

print("Dependencies loaded successfully")

Dependencies loaded successfully


In [2]:
# Import the reusable experiment runner. It keeps the notebook focused on
# the test plan while keeping source-copy and cleanup logic in one module.
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'include').exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'include'))

from observability_experiments import run_all_experiments, write_report

In [8]:
# Run experiments 1–10 for both ETL and ELT. The returned DataFrame has
# one row per DAG per experiment and contains all four required fields.
report = run_all_experiments()
display(report)

,experiment,dag,expected,observed,where_seen,what_to_change
0,1,ETL,Both DAG implementations complete and publish ...,"{'run_id': 'step10-exp01-etl', 'rows_extracted...","Local pipeline return value, DuckDB audit/qual...",Keep the happy-path audit and row-count checks.
1,1,ELT,Both DAG implementations complete and publish ...,{'warehouse_path': '/var/folders/__/qj33dz2j59...,"Local pipeline return value, DuckDB audit/qual...",Keep the happy-path audit and row-count checks.
2,2,ETL,Fact row count does not change after a rerun.,1851 -> 1851,"Local pipeline return value, DuckDB audit/qual...",Keep replace/upsert semantics and add an autom...
3,2,ELT,Raw and staging counts remain stable for a bas...,2412 -> 2412; staging 2400 -> 2400,"Local pipeline return value, DuckDB audit/qual...",Keep raw replacement for a rerun and use appen...
4,3,ETL,"150 new orders inserted, 10 corrected orders u...","inserted=150, updated=10, fact_rows=1982","Local pipeline return value, DuckDB audit/qual...",Keep the order_id upsert and monitor inserted/...
5,3,ELT,"150 new orders inserted, 10 corrected orders u...","inserted=150, updated=10, raw_rows=2572, stage...","Local pipeline return value, DuckDB audit/qual...",Keep append-only raw storage and latest-row SQ...
6,4,ETL,Both pipelines fail loudly on the missing prod...,FileNotFoundError: [Errno 2] No such file or d...,"Local pipeline return value, DuckDB audit/qual...",Add a source-file preflight check and alert on...
7,4,ELT,Both pipelines fail loudly on the missing prod...,FileNotFoundError: [Errno 2] No such file or d...,"Local pipeline return value, DuckDB audit/qual...",Add the same preflight check before raw loading.
8,5,ETL,The five added rows are rejected or marked FAI...,"quarantined=208, issue_rows=531","Local pipeline return value, DuckDB audit/qual...",Keep row-level reasons and add assertions for ...
9,5,ELT,The five added rows are rejected or marked FAI...,"passing_staging=2318, failed_staging=87","Local pipeline return value, DuckDB audit/qual...",Expose failure_reason and dq_status in the sta...


In [4]:
# Save the complete experiment matrix as repository artifacts for review.
csv_path, markdown_path = write_report(report, REPO_ROOT / 'assignment')
print(f'CSV report: {csv_path}')
print(f'Markdown report: {markdown_path}')
print(f'Rows written: {len(report)} (10 experiments × 2 DAGs)')

ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

In [ ]:
# Confirm that every experiment has both DAG results and every required
# documentation field is populated before the notebook is considered done.
assert set(report['experiment']) == set(range(1, 11))
assert set(report['dag']) == {'ETL', 'ELT'}
for column in ['expected', 'observed', 'where_seen', 'what_to_change']:
    assert report[column].notna().all()
print('Step 10 validation passed: 20 documented experiment results are present.')

## Important interpretation

Experiments 4, 6, and 10 also require the Airflow UI to confirm retry states, task states, and Clear-based recovery. This notebook validates the same pipeline implementations locally and records the expected Airflow observation in the report. Run the DAGs in Airflow to attach screenshots or task-log evidence.